### Hyperparameter tuning - Fine tune the model (optimize the model for better output)

1. GridSearchCV - Try multiple combination
2. KerasClassificer - Keras model for sklearn
3. Pipeline - getting the output in structured workflow

In [102]:
## Import libraries

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle
import numpy as np

In [39]:
data = pd.read_csv("F:\GEN AI\Gen AI\PROJECTS\Deep Lerning\ANN-bank-customer\data\Churn_Modelling.csv.csv")

In [40]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [41]:
## Map gender to 1 and 0
data['Gender'] = data['Gender'].map({'Male': 1, 'Female': 0})

In [42]:
onehot_geo = OneHotEncoder()
geo_encoded = onehot_geo.fit_transform(data[['Geography']]).toarray()

In [43]:
geo_encoded_df = pd.DataFrame(geo_encoded, 
                              columns=onehot_geo.get_feature_names_out(['Geography']))

In [44]:
data = pd.concat([data.drop(['Geography'], axis=1), geo_encoded_df], axis=1)

In [45]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [46]:
## Divide the data into features and target
X = data.drop(columns=['Exited'])
y = data['Exited']

In [47]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [48]:
X_train.shape, X_test.shape

((8000, 12), (2000, 12))

In [49]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Hyper parameter tuning
Write a function to create the model and try different parameters

layer = 1 -> no loops

layer = 2 -> add extra 1 layer

layer = 3 -> add extra 2 layers

This will make the architecture more flexible

In [50]:
def create_model(neurons=32, layers =1):
    model = Sequential()  ## start te neural network
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))  ## add the first hidden layer with input shape

    ## Add more layers to this dynamically
    for _ in range(layers-1):   ## _ is a throwaway variable(python convention) used when we need to loop a specific number of times but dont know the exact value
        model.add(Dense(neurons, activation='relu'))  ## add more hidden layers
    
    model.add(Dense(1, activation='sigmoid'))  ## add the output layer
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [51]:
## Create a keras classifier

model = KerasClassifier(layers = 1, neurons=32, build_fn=create_model, verbose=1)

In [52]:
## Define the hyperparameter grid
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2, 3],
    'epochs': [50, 100, 150]
}

Based on the above parameter grids, we will try different combinations

In [53]:
grid = GridSearchCV(estimator=model, 
                    param_grid=param_grid, 
                    n_jobs=-1, ## Uses all the CPU cores (faster)
                    cv=3,  ## 3-fold cross-validation, Data will split in 3 parts: training, validation, and test
                    verbose=1)  ## Shows the progress

In [54]:
grid_result = grid.fit(X_train, y_train)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Epoch 1/100


f:\GEN AI\Gen AI\PROJECTS\Deep Lerning\ANN-bank-customer\.venv\lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
f:\GEN AI\Gen AI\PROJECTS\Deep Lerning\ANN-bank-customer\.venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8001 - loss: 0.4656
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8165 - loss: 0.4290
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8225 - loss: 0.4164
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8269 - loss: 0.4076
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8329 - loss: 0.3997
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8379 - loss: 0.3914
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8428 - loss: 0.3827
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8460 - loss: 0.3744
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8491 - loss: 0.3670
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8508 - loss: 0.3616
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8536 - loss: 0.3567
Epoch 12/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

In [76]:
best_model = grid_result.best_estimator_

In [55]:
print("Best: %f using %s" %(grid_result.best_score_, grid_result.best_params_))

Best: 0.857749 using {'epochs': 100, 'layers': 1, 'neurons': 16}


In [81]:
## New input data

new_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Female',
    'Age': 40,
    'Tenure': 3,
    'Balance': 50000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 60000,
}

In [82]:
new_data_df = pd.DataFrame([new_data])

In [83]:
new_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Female,40,3,50000,2,1,1,60000


In [84]:
with open('F:\GEN AI\Gen AI\PROJECTS\Deep Lerning\ANN-bank-customer\encodings and scaling\geo_encoded.pkl', 'rb') as f:
    geo_encoded = pickle.load(f)

with open('F:\GEN AI\Gen AI\PROJECTS\Deep Lerning\ANN-bank-customer\encodings and scaling\gender_encoded.pkl', 'rb') as f:
    gender_encoded = pickle.load(f)

In [85]:
## Use the previous stored encoding and encode the Gender from new input data
new_data_df['Gender'] = new_data_df['Gender'].map(gender_encoded)

In [86]:
geo_data = geo_encoded.transform(
    new_data_df[['Geography']]
).toarray()

In [87]:
geo_df = pd.DataFrame(
    geo_data,
    columns=geo_encoded.get_feature_names_out(['Geography'])
)

In [88]:
new_data_df = pd.concat(
    [new_data_df.drop('Geography', axis=1), geo_df],
    axis=1
)

In [89]:
new_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,0,40,3,50000,2,1,1,60000,1.0,0.0,0.0


In [90]:
## Load the scaler

with open('../encodings and scaling/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [91]:
new_data_df_scaled = scaler.transform(new_data_df)

In [92]:
new_data_df_scaled

array([[-0.53598516, -1.09499335,  0.10479359, -0.69539349, -0.41792108,
         0.80843615,  0.64920267,  0.97481699, -0.70296551,  1.00150113,
        -0.57946723, -0.57638802]])

Now our data is encoded and scaled using the same encodings and scalings which we have used to train our data. Basically it means our model accuracy will be better

In [93]:
prediction = best_model.predict(new_data_df_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step


In [94]:
prediction

array([0])

In [96]:
prediction_prob = best_model.predict_proba(new_data_df_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step


In [103]:
prediction_prob = np.array(prediction_prob)

In [104]:
# Extract probability safely
if prediction_prob.ndim == 2:
    prob = prediction_prob[0][0]
else:
    prob = prediction_prob[0]

In [105]:
## Decision logic 

if prob > 0.5:
    print("Customer is likely to churn.")
else:
    print("Customer is not likely to churn.")

Customer is likely to churn.
